In [ ]:
!pip install jams >/dev/null 2>&1

In [1]:
import numpy as np
from collections import defaultdict, Counter
import jams
from random import randint
from pathlib import Path
from typing import Dict, List, Tuple, Any


# Load chord sequences from guitarset

In [3]:
# Root directory for annotations
annotation_dir = Path(r'C:\Users\Luis\Desktop\Master\DSAP\chord-detection-project\guitarset\annotation')

def extract_chord_progression(jam_path: Path) -> List[str]:
    """Extract chord progression from a JAMS file.
    
    Returns a list of chord labels (strings).
    """
    jam = jams.load(str(jam_path))
    chord_anns = jam.search(namespace="chord")
    
    if not chord_anns:
        return []
    
    return [event.value for event in chord_anns[0].data]

# Collect chord progressions as a list of lists
chord_progressions = []

# Get all _comp.jams files
comp_files = sorted([f for f in annotation_dir.glob("*.jams") if f.stem.endswith("_comp")])
print(f"Found {len(comp_files)} accompaniment files\n")

for jam_path in comp_files:
    progression = extract_chord_progression(jam_path)
    chord_progressions.append(progression)

Found 180 accompaniment files



In [4]:
def standardize_chord(chord_label: str) -> str:
    """Convert chord label to standardized format.
    
    Rules:
    - Major chords (X:maj) → X (root only)
    - Minor chords (X:min) → Xm
    - Seventh chords (X:7) → X (treated as major)
    - Half-diminished 7th (X:hdim7) → Xm (treated as minor)
    - Inversions (/N) are ignored
    """
    # Remove inversion if present
    chord_label = chord_label.split("/")[0]
    
    if ":" not in chord_label:
        return chord_label  # Return as-is if malformed
    
    root, quality = chord_label.split(":", 1)
    
    # Map quality to standardized format
    if quality == "maj":
        return root  # Major → just root
    elif quality == "min":
        return f"{root}m"  # Minor → root + "m"
    elif quality == "7":
        return f"{root}7"  # Dominant 7th → major (just root)
    elif quality == "hdim7":
        return f"{root}hdim7"  # Half-diminished → minor
    else:
        # Default: treat unknown as major
        return root

# Apply standardization to all progressions
chord_progressions_std = [
    [standardize_chord(c) for c in prog] 
    for prog in chord_progressions
]

In [5]:
# Display all 180 standardized chord progressions
print(f"All {len(chord_progressions_std)} Standardized Chord Progressions")
print("=" * 100)

for i, progression in enumerate(chord_progressions_std):
    chord_seq = " → ".join(progression)
    print(f"\n[{i:3d}] ({len(progression):2d} chords): {chord_seq}")


All 180 Standardized Chord Progressions

[  0] ( 6 chords): D# → G# → D# → A# → G# → D#

[  1] ( 6 chords): F# → B → F# → C# → B → F#

[  2] (14 chords): Em → A7 → D → G → C#hdim7 → F#7 → Bm → Em → A7 → D → G → C#hdim7 → F#7 → Bm

[  3] (14 chords): C#m → F#7 → B → E → A#hdim7 → D#7 → G#m → C#m → F#7 → B → E → A#hdim7 → D#7 → G#m

[  4] (16 chords): G → D → Em → Bm → C → G → C → D → G → D → Em → Bm → C → G → C → D

[  5] (16 chords): E → B → C#m → G#m → A → E → A → B → E → B → C#m → G#m → A → E → A → B

[  6] ( 6 chords): G# → C# → G# → D# → C# → G#

[  7] ( 6 chords): C → F → C → G → F → C

[  8] (14 chords): G#m → C#7 → F# → B → Fhdim7 → A#7 → D#m → G#m → C#7 → F# → B → Fhdim7 → A#7 → D#m

[  9] (14 chords): Cm → F7 → A# → D# → Ahdim7 → D7 → Gm → Cm → F7 → A# → D# → Ahdim7 → D7 → Gm

[ 10] (16 chords): C# → G# → A#m → Fm → F# → C# → F# → G# → C# → G# → A#m → Fm → F# → C# → F# → G#

[ 11] (16 chords): A → E → F#m → C#m → D → A → D → E → A → E → F#m → C#m → D → A → D → E

[ 12] ( 6 cho

# N Gram Model

In [6]:
class NGramModel:
    def __init__(self, n=3):
        self.N = n
        # We need counts for ALL orders (N, N-1, ... 1) for JM smoothing
        # structure: self.counts[order][history_tuple][chord] = count
        self.counts = {i: defaultdict[tuple, Counter](Counter) for i in range(1, n + 1)}
        self.total_counts = {i: defaultdict[Any, int](int) for i in range(1, n + 1)}
        self.vocab = set[Any]()

    def train(self, sequences):
        """
        Counts occurrences of chords in the dataset.
        """
        for seq in sequences:
            # Update vocabulary
            self.vocab.update(seq)
            
            # Count for every order from 1 to N
            for order in range(1, self.N + 1):
                for i in range(len(seq) - order + 1):
                    ngram = tuple(seq[i : i+order])
                    history = ngram[:-1]
                    target = ngram[-1]
                    
                    self.counts[order][history][target] += 1
                    self.total_counts[order][history] += 1
        
        print(f"Training complete. Vocab size: {len(self.vocab)}")

    def get_ml_prob(self, history, chord):
        """
        Calculates Maximum Likelihood Probability.
        P_ML = count(history + chord) / count(history)
        """
        order = len(history) + 1
        if order > self.N: return 0
        
        hist_count = self.total_counts[order][history]
        if hist_count == 0:
            return 0.0
        
        return self.counts[order][history][chord] / hist_count

    # --- 3. Jelinek-Mercer Smoothing Implementation ---

    def get_jm_prob(self, history, chord, lambdas):
        """
        Recursive Interpolation.
        P_JM(C|History) = lambda * P_ML(C|History) + (1-lambda) * P_JM(C|Shorter_History)
        """
        # Ensure history matches the model order
        history = tuple(history)[-(self.N-1):]
        order = len(history) + 1
        
        # Get the lambda for this specific order (e.g., lambda_2 for Trigrams)
        # Note: lambdas list should be [lambda_0, lambda_1, lambda_2...]
        lam = lambdas[order - 1] if order - 1 < len(lambdas) else 0.5
        
        # 1. Calculate Maximum Likelihood for this level
        p_ml = self.get_ml_prob(history, chord)
        
        # 2. Recursive Step
        if order == 1:
            # Base Case (Order 0): Uniform Distribution (1 / Vocabulary Size)
            # The paper mentions this in Section 2.2
            p_lower = 1.0 / len(self.vocab) if self.vocab else 1e-6
        else:
            # Recursive call with shortened history (Backoff)
            shorter_history = history[1:]
            p_lower = self.get_jm_prob(shorter_history, chord, lambdas)
            
        # 3. Combine
        return (lam * p_ml) + ((1 - lam) * p_lower)

    def predict_next(self, input_sequence, lambdas=None, top_k=5, use_smoothing=True):
        """
        Predicts the most likely next chord(s) given an input sequence.
        
        Args:
            input_sequence: List of chords (e.g., ['C:maj', 'G:maj'])
            lambdas: Lambda values for JM smoothing (default: [0.1, 0.5, 0.9])
            top_k: Number of top predictions to return
            use_smoothing: Whether to use JM smoothing (True) or raw ML (False)
        
        Returns:
            List of tuples: [(chord, probability), ...]
        """
        if lambdas is None:
            lambdas = [0.1, 0.5, 0.9]
        
        # Extract the relevant history (last N-1 chords)
        history = tuple(input_sequence)[-(self.N - 1):]
        
        # Calculate probability for each chord in vocabulary
        predictions = []
        for chord in self.vocab:
            if use_smoothing:
                prob = self.get_jm_prob(history, chord, lambdas)
            else:
                prob = self.get_ml_prob(history, chord)
            predictions.append((chord, prob))
        
        # Sort by probability (descending) and return top_k
        predictions.sort(key=lambda x: x[1], reverse=True)
        return predictions[:top_k]

In [8]:
# 1. Load Data
sequences = chord_progressions_std # Use this for real data
# sequences = get_dummy_data()    # Use this for testing now

# 2. Initialize and Train
N = 3 # Trigram model
model = NGramModel(n=N)
model.train(sequences)

# 3. Test with a sequence (Applying Smoothing)

# Let's say user played ['C:maj', 'F:maj']
# We want to know probability of 'G:maj' coming next
history = ('F', 'C')
candidate = 'G'

# Define Lambdas (Hyperparameters)
# The paper optimizes these, but we can start with 0.5
# [lambda_0 (unigram), lambda_1 (bigram), lambda_2 (trigram)]
my_lambdas = [0.1, 0.5, 0.9] 

# A. Try Standard ML (might be zero if sequence unseen)
prob_ml = model.get_ml_prob(history, candidate)

# B. Try JM Smoothing
prob_jm = model.get_jm_prob(history, candidate, my_lambdas)

print(f"\nContext: {history}")
print(f"Next Chord Candidate: {candidate}")
print(f"Standard Probability (ML): {prob_ml:.4f}")
print(f"Smoothed Probability (JM): {prob_jm:.4f}")

# Demonstrate robustness: Try a weird chord 'X:min' not in dataset
# ML will be 0.0. JM should be non-zero (due to uniform base case).
prob_weird = model.get_jm_prob(history, 'X:min', my_lambdas)
print(f"Probability of unknown chord 'X:min': {prob_weird:.6f}")

Training complete. Vocab size: 42

Context: ('F', 'C')
Next Chord Candidate: G
Standard Probability (ML): 0.1429
Smoothed Probability (JM): 0.1549
Probability of unknown chord 'X:min': 0.001071


In [9]:
print(f'Model vocab size: {len(model.vocab)}')
print(f'Model vocab: {model.vocab}')

Model vocab size: 42
Model vocab: {'Am', 'D#m', 'Em', 'Fhdim7', 'F7', 'A', 'Fm', 'D', 'Ghdim7', 'Ehdim7', 'C', 'E', 'A7', 'Bm', 'D#7', 'C#m', 'Gm', 'C#hdim7', 'A#hdim7', 'F', 'Ahdim7', 'C7', 'C#', 'C#7', 'F#m', 'G#', 'G#m', 'E7', 'A#', 'F#7', 'D7', 'G#hdim7', 'G', 'A#m', 'Chdim7', 'F#', 'Cm', 'Dm', 'B', 'A#7', 'D#', 'G#7'}


In [10]:
# ============================================
# CHORD PREDICTION EXAMPLES
# ============================================

# Example 1: Predict next chord given an input sequence
print("=" * 50)
print("EXAMPLE 1: Predict next chord from input sequence")
print("=" * 50)

input_sequence = ['C', 'G']
print(f"\nInput sequence: {input_sequence}")
print(f"\nTop 5 predicted next chords:")

predictions = model.predict_next(input_sequence, top_k=5)
for i, (chord, prob) in enumerate(predictions, 1):
    print(f"  {i}. {chord:15} (probability: {prob:.4f})")

# Example 2: Try different input sequences
print("\n" + "=" * 50)
print("EXAMPLE 2: Different input sequences")
print("=" * 50)

test_sequences = [
    ['Am', 'F'],
    ['G', 'D'],
    ['Em', 'C'],
    ['F', 'G']
]

for seq in test_sequences:
    print(f"\nInput: {seq}")
    preds = model.predict_next(seq, top_k=3)
    print(f"  Top 3 predictions: {[(c, f'{p:.4f}') for c, p in preds]}")


EXAMPLE 1: Predict next chord from input sequence

Input sequence: ['C', 'G']

Top 5 predicted next chords:
  1. Am              (probability: 0.6178)
  2. C               (probability: 0.2152)
  3. F               (probability: 0.1042)
  4. D               (probability: 0.0124)
  5. C#hdim7         (probability: 0.0067)

EXAMPLE 2: Different input sequences

Input: ['Am', 'F']
  Top 3 predictions: [('C', '0.0195'), ('G', '0.0150'), ('A#', '0.0104')]

Input: ['G', 'D']
  Top 3 predictions: [('Em', '0.6079'), ('A', '0.3180'), ('G', '0.0147')]

Input: ['Em', 'C']
  Top 3 predictions: [('G', '0.0264'), ('F', '0.0208'), ('D', '0.0069')]

Input: ['F', 'G']
  Top 3 predictions: [('C', '0.9152'), ('Am', '0.0178'), ('D', '0.0124')]
